# Step10f — Perfiles Automáticos (24 provincias + 221 cantones)

**Proyecto:** EFCI Ecuador (ENSO Flood Chronicity Index)
**Fase:** 10 — Productos, sección E: perfiles por unidad administrativa (Fase 0 §9, §14)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

**Objetivo de este notebook:** producir un documento PDF consolidado
(`outputs/reports/Perfiles_EFCI_Ecuador.pdf`) con una página por unidad
administrativa — **24 provincias + 221 cantones = 245 páginas** —
cada una con sus indicadores clave (EFCI, 4 sub-índices, ranking,
impactos históricos, clasificación de hotspot cuando aplica).

**decisión confirmada por el equipo (AskUserQuestion, 2026-08-06):**
cobertura completa (24 + 221), no solo una muestra de los cantones más
crónicos — para que cualquier cantón, tenga o no historial relevante,
tenga su propia página de referencia.

**Estructura del documento:** agrupado por provincia — la página de
resumen de cada provincia va seguida inmediatamente de las páginas de
sus cantones, en vez de dos bloques separados (24 provincias + 221
cantones) — más fácil de navegar para un lector que busca "Manabí y
sus cantones", no "todas las provincias, luego todos los cantones sin
agrupar".

**English:** Automatic profiles: one page per province (24) and per canton (221), plus a cover, in a single PDF.

## 0. Configuración del entorno

**English:** Environment setup.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper that creates folders when missing
from src.utilidades.io_utils import ensure_dir

# Tabular data
import pandas as pd
# Numerical arrays
import numpy as np
# Vector geodata
import geopandas as gpd
# Plotting library
import matplotlib.pyplot as plt
# Multi-page PDF writer
from matplotlib.backends.backend_pdf import PdfPages

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10f_profiles", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10f iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Output folder for the reports
reports_dir = ensure_dir(PROJECT_ROOT / "outputs" / "reports")
# Default font size
plt.rcParams["font.size"] = 8

## 1. Datos: rankings, sub-índices, impactos, hotspots

**English:** Data: rankings, component indices, impacts, hotspots.

In [ ]:
# Read the master canton dataset (no geometry needed)
master = gpd.read_file(processed_dir / "mapa_master_canton.gpkg").drop(columns="geometry")
# Rank cantons by chronicity index
master = master.sort_values("efci", ascending=False).reset_index(drop=True)
# National rank of each canton
master["ranking_nacional"] = range(1, len(master) + 1)

# Read the province-level index
efci_prov = pd.read_csv(processed_dir / "efci_composite_provincia.csv")
# Combined view, sorted by index
efci_prov = efci_prov[efci_prov["hazard_type"] == "TODOS"].sort_values("efci", ascending=False).reset_index(drop=True)
# National rank of each province
efci_prov["ranking_nacional"] = range(1, len(efci_prov) + 1)

# Read the georeferenced events
he = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Official names
canton_names = gpd.read_file(processed_dir / "adm2_canton.geojson")[["canton_code", "canton_name_official"]]
canton_names["canton_code"] = canton_names["canton_code"].astype(str)
he["canton_code"] = he["canton_code"].astype(str)
# Attach official names to each event
he = he.merge(canton_names, on="canton_code", how="left")

# Events per canton and year
eventos_por_anio_canton = he.groupby(["canton_code", "year"]).size().rename("n").reset_index()
# Events per province and year
eventos_por_anio_provincia = he.groupby(["province_name_official", "year"]).size().rename("n").reset_index()

# Years present in the data
ANIOS = sorted(he["year"].unique().tolist())
logger.info("Insumos de perfiles listos: %d cantones, %d provincias, %d años", len(master), len(efci_prov), len(ANIOS))

## 2. Funciones de dibujo de una página de perfil

**English:** Functions that draw one profile page.

In [ ]:
# Colour of each chronicity class
COLOR_EFCI = {"Muy Baja": "#fee5d9", "Baja": "#fcae91", "Moderada": "#fb6a4a", "Alta": "#de2d26", "Muy Alta": "#a50f15"}
# Component indices shown in every profile (column, label)
SUB_INDICES = [
    ("severity_index", "Severidad"), ("recurrence_index", "Recurrencia"),
    ("persistence_index", "Persistencia"), ("enso_magnitude_mean", "Magnitud ENSO"),
]


# Draw the page header: name, class badge and national rank
def dibujar_encabezado(fig, titulo, subtitulo, clase_efci, efci_valor, ranking_texto):
    fig.text(0.06, 0.94, titulo, fontsize=16, fontweight="bold")
    fig.text(0.06, 0.905, subtitulo, fontsize=10, color="dimgray")
    color = COLOR_EFCI.get(clase_efci, "#cccccc")
    fig.patches.append(plt.Rectangle((0.75, 0.89), 0.19, 0.07, transform=fig.transFigure,
                                      facecolor=color, edgecolor="black", linewidth=0.5))
    fig.text(0.845, 0.935, f"EFCI {efci_valor:.3f}", ha="center", fontsize=11, fontweight="bold")
    fig.text(0.845, 0.905, clase_efci, ha="center", fontsize=9)
    fig.text(0.06, 0.865, ranking_texto, fontsize=9, style="italic", color="#1f4e78")


# Horizontal bars with the four component indices
def dibujar_subindices(ax, fila):
    valores = [fila[col] for col, _ in SUB_INDICES]
    etiquetas = [lbl for _, lbl in SUB_INDICES]
    ax.barh(etiquetas, valores, color="#1f4e78")
    ax.set_xlim(0, 1)
    ax.set_title("Sub-índices EFCI (0-1)", fontsize=9, loc="left")
    for i, v in enumerate(valores):
        ax.text(v + 0.02, i, f"{v:.3f}", va="center", fontsize=7)


# Bar chart of events per year
def dibujar_tendencia_anual(ax, serie_anual):
    # Show every year, even with zero events
    serie_anual = serie_anual.reindex(ANIOS, fill_value=0)
    ax.bar(serie_anual.index.astype(str), serie_anual.values, color="#6baed6")
    ax.set_title("Eventos por año", fontsize=9, loc="left")
    ax.tick_params(axis="x", rotation=90, labelsize=6)
    ax.tick_params(axis="y", labelsize=7)


# Draw a two-column indicator table
def dibujar_tabla_impactos(ax, filas_tabla):
    ax.axis("off")
    # Create the table
    tabla = ax.table(cellText=filas_tabla, colLabels=["Indicador", "Valor"], loc="center", cellLoc="left")
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(7.5)
    tabla.scale(1, 1.4)
    # Style the header row
    for (row, col), cell in tabla.get_celld().items():
        if row == 0:
            cell.set_facecolor("#1f4e78")
            cell.set_text_props(color="white", fontweight="bold")

## 3. Página de perfil — Provincia

**English:** Province profile page.

In [ ]:
# One-page profile of a province
def pagina_provincia(pdf, fila):
    fig = plt.figure(figsize=(8.5, 11))
    dibujar_encabezado(
        fig, fila["province_name_official"], "Perfil de Provincia — EFCI Ecuador",
        fila["clase_efci"], fila["efci"],
        f"Ranking nacional (provincias): {fila['ranking_nacional']} de 24",
    )

    # Component-index panel
    ax1 = fig.add_axes([0.08, 0.55, 0.4, 0.28])
    dibujar_subindices(ax1, fila)

    # Annual-trend panel
    ax2 = fig.add_axes([0.55, 0.55, 0.4, 0.28])
    serie = eventos_por_anio_provincia[eventos_por_anio_provincia["province_name_official"] == fila["province_name_official"]] \
        .set_index("year")["n"]
    dibujar_tendencia_anual(ax2, serie)

    # Indicator table panel
    ax3 = fig.add_axes([0.08, 0.1, 0.84, 0.35])
    # Rows of the indicator table
    filas_tabla = [
        ["Nº de eventos (2015-2025)", f"{int(fila['n_eventos']):,}"],
        ["Clase de cronicidad EFCI", fila["clase_efci"]],
        ["Índice de Severidad", f"{fila['severity_index']:.3f}"],
        ["Índice de Recurrencia", f"{fila['recurrence_index']:.3f}"],
        ["Índice de Persistencia", f"{fila['persistence_index']:.3f}"],
        ["Magnitud ENSO (media)", f"{fila['enso_magnitude_mean']:.3f}"],
    ]
    dibujar_tabla_impactos(ax3, filas_tabla)

    fig.text(0.06, 0.02, "EFCI Ecuador · FAO OER · Perfil generado 2026-08-06 · Fuente: docs/methodology (Fases 1-9)",
              fontsize=6, color="gray")
    # Save the page
    pdf.savefig(fig)
    plt.close(fig)

## 4. Página de perfil — Cantón

**English:** Canton profile page.

In [ ]:
# One-page profile of a canton
def pagina_canton(pdf, fila):
    fig = plt.figure(figsize=(8.5, 11))
    dibujar_encabezado(
        fig, fila["canton_name_official"], f"Perfil de Cantón — Provincia de {fila['province_name_official']}",
        fila["clase_efci"], fila["efci"],
        f"Ranking nacional (cantones): {fila['ranking_nacional']} de 221",
    )

    ax1 = fig.add_axes([0.08, 0.55, 0.4, 0.28])
    dibujar_subindices(ax1, fila)

    ax2 = fig.add_axes([0.55, 0.55, 0.4, 0.28])
    serie = eventos_por_anio_canton[eventos_por_anio_canton["canton_code"] == fila["canton_code"]].set_index("year")["n"]
    dibujar_tendencia_anual(ax2, serie)

    ax3 = fig.add_axes([0.08, 0.1, 0.84, 0.35])
    filas_tabla = [
        ["Nº de eventos (2015-2025)", f"{int(fila['frecuencia_inundaciones']):,}"],
        ["Años afectados", f"{int(fila['anios_afectados'])} de 11"],
        ["Meses en El Niño con evento", f"{int(fila['frecuencia_enso']):,}"],
        ["Clasificación Hotspot (Getis-Ord Gi*)", fila["gi_clase_efci"]],
        ["Población afectada (máx. histórico)", f"{int(fila['poblacion_total']):,}"],
        ["Pérdidas agrícolas (máx. histórico, ha)", f"{fila['agricultura_total']:,.1f}"],
        ["Viviendas afectadas/destruidas (máx. histórico)", f"{int(fila['vivienda_total']):,}"],
    ]
    dibujar_tabla_impactos(ax3, filas_tabla)

    fig.text(0.06, 0.02, "EFCI Ecuador · FAO OER · Perfil generado 2026-08-06 · Fuente: docs/methodology (Fases 1-9)",
              fontsize=6, color="gray")
    pdf.savefig(fig)
    plt.close(fig)

## 5. Portada y ensamblado del documento completo (245 páginas)

**English:** Cover and assembly of the full document (246 pages).

In [ ]:
# Output PDF with all profiles
perfiles_path = reports_dir / "Perfiles_EFCI_Ecuador.pdf"

# Write cover, then each province followed by its cantons
with PdfPages(perfiles_path) as pdf:
    # Portada
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis("off")
    ax.text(0.5, 0.65, "Perfiles de Cronicidad de\nInundaciones por El Niño", ha="center", fontsize=22, fontweight="bold")
    ax.text(0.5, 0.55, "24 provincias + 221 cantones", ha="center", fontsize=14)
    ax.text(0.5, 0.48, "Ecuador 2015–2025 · EFCI", ha="center", fontsize=12, style="italic")
    ax.text(0.5, 0.15, "FAO — Oficina de Emergencias y Resiliencia (OER)\nJerson Achicanoy · 2026-08-06", ha="center", fontsize=10)
    pdf.savefig(fig)
    plt.close(fig)

    n_paginas = 1
    # Loop over provinces in ranking order
    for _, fila_prov in efci_prov.iterrows():
        pagina_provincia(pdf, fila_prov)
        n_paginas += 1
        # Cantons of the province, highest chronicity first
        cantones_de_provincia = master[master["province_name_official"] == fila_prov["province_name_official"]] \
            .sort_values("efci", ascending=False)
        # One page per canton
        for _, fila_canton in cantones_de_provincia.iterrows():
            pagina_canton(pdf, fila_canton)
            n_paginas += 1

    logger.info("Documento de perfiles generado: %s (%d páginas)", perfiles_path, n_paginas)

print(f"Perfiles exportados: {perfiles_path}")

## 6. Validación

**English:** Validation.

In [ ]:
# Check the PDF was written
assert perfiles_path.exists() and perfiles_path.stat().st_size > 0
# Check cover + 24 provinces + 221 cantons
assert n_paginas == 1 + 24 + 221, f"se esperaban 246 páginas (portada+24+221), hubo {n_paginas}"
logger.info("Fase 10f — validación OK. %d páginas generadas.", n_paginas)
print("Fase 10f completa.")